# KiwiTel Customer Churn Analysis
### Full EDA + Feature Engineering + Predictive Modelling

**Company:** KiwiTel (fictional NZ telecom)  
**Dataset:** 2,000 customer records  
**Goal:** Identify churn drivers and build a predictive model (AUC-ROC > 0.75)

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

from src.features import engineer_features, get_feature_columns

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/raw/customers.csv')
print(f'Shape: {df.shape}')
print(f'Churn rate: {(df["churn"] == "Yes").mean():.1%}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())

## 2. Exploratory Data Analysis

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_counts = df['churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#2ECC71', '#E74C3C'])
axes[0].set_title('Churn Distribution')

churn_by_contract = df.groupby('contract_type')['churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).sort_values(ascending=False)
churn_by_contract.plot(kind='bar', ax=axes[1], color=['#E74C3C', '#F39C12', '#2ECC71'])
axes[1].set_title('Churn Rate by Contract Type (%)')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly charges vs tenure coloured by churn
fig = px.scatter(
    df, x='tenure_months', y='monthly_charges', color='churn',
    color_discrete_map={'Yes': '#E74C3C', 'No': '#2ECC71'},
    title='Monthly Charges vs Tenure by Churn Status',
    labels={'tenure_months': 'Tenure (months)', 'monthly_charges': 'Monthly Charges (NZD)'},
    opacity=0.6,
)
fig.show()

In [ ]:
# Churn rate by support calls
support_churn = df.groupby('num_support_calls')['churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
support_churn.columns = ['num_support_calls', 'churn_rate_pct']

fig = px.bar(
    support_churn, x='num_support_calls', y='churn_rate_pct',
    title='Churn Rate by Number of Support Calls',
    labels={'num_support_calls': 'Support Calls', 'churn_rate_pct': 'Churn Rate (%)'},
    color='churn_rate_pct', color_continuous_scale=['#2ECC71', '#E74C3C'],
)
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
# Correlation heatmap (numeric features)
num_cols = ['age', 'tenure_months', 'monthly_charges', 'total_charges',
            'num_support_calls', 'late_payments']
df_num = df[num_cols].copy()
df_num['churn_bin'] = (df['churn'] == 'Yes').astype(int)

plt.figure(figsize=(8, 6))
sns.heatmap(df_num.corr(), annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. SQL Analysis

In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, index=False)

# Query 1: Overall churn rate
q1 = pd.read_sql("""
    SELECT
        COUNT(*) AS total_customers,
        SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct
    FROM customers
""", conn)
print('Query 1: Overall Churn Rate')
display(q1)

In [ ]:
# Query 2: Churn rate by contract type
q2 = pd.read_sql("""
    SELECT contract_type,
           COUNT(*) AS total,
           SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
           ROUND(100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct
    FROM customers GROUP BY contract_type ORDER BY churn_rate_pct DESC
""", conn)
print('Query 2: Churn Rate by Contract Type')
display(q2)

In [ ]:
# Query 5: Churn by support call bucket
q5 = pd.read_sql("""
    SELECT
        CASE WHEN num_support_calls = 0 THEN '0 calls'
             WHEN num_support_calls BETWEEN 1 AND 2 THEN '1-2 calls'
             WHEN num_support_calls BETWEEN 3 AND 4 THEN '3-4 calls'
             ELSE '5+ calls' END AS support_bucket,
        COUNT(*) AS total,
        SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(100.0 * SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct
    FROM customers GROUP BY support_bucket
""", conn)
print('Query 5: Churn by Support Call Bucket')
display(q5)

In [ ]:
# Query 7: Revenue at risk
q7 = pd.read_sql("""
    SELECT churn, COUNT(*) AS customers,
           ROUND(SUM(monthly_charges), 2) AS total_monthly_revenue_nzd,
           ROUND(SUM(monthly_charges) * 12, 2) AS annualised_revenue_nzd
    FROM customers GROUP BY churn ORDER BY churn DESC
""", conn)
print('Query 7: Revenue at Risk')
display(q7)
conn.close()

## 4. Feature Engineering

In [ ]:
df_feat = engineer_features(df)
print('New features added:')
new_cols = [c for c in df_feat.columns if c not in df.columns]
print(new_cols)
df_feat[new_cols + ['churn']].head()

## 5. Model Training & Evaluation

In [ ]:
features = get_feature_columns()
X = df_feat[features]
y = (df['churn'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.1%}, Test: {y_test.mean():.1%}')

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]
print('LOGISTIC REGRESSION')
print(classification_report(y_test, lr_pred, target_names=['Retained', 'Churned']))
print(f'AUC-ROC: {roc_auc_score(y_test, lr_proba):.4f}')

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_proba)
print('RANDOM FOREST')
print(classification_report(y_test, rf_pred, target_names=['Retained', 'Churned']))
print(f'AUC-ROC: {rf_auc:.4f}')

In [ ]:
# ROC Curve comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name, proba in [('Logistic Regression', lr_proba), ('Random Forest', rf_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', label='Random baseline')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

# Feature importance
importances = pd.Series(rf.feature_importances_, index=features).nlargest(10)
importances.sort_values().plot(kind='barh', ax=axes[1], color='#3498DB')
axes[1].set_title('Top 10 Feature Importances (Random Forest)')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print(f'\nTarget AUC-ROC > 0.75: {"PASS" if rf_auc > 0.75 else "FAIL"} ({rf_auc:.4f})')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn_r',
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'])
plt.title('Random Forest — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 6. Conclusions

### Key Churn Drivers
1. **Contract type** — month-to-month contracts have the highest churn rate
2. **Support calls** — 3+ calls dramatically increases churn probability  
3. **Tenure** — new customers (< 12 months) are the most vulnerable
4. **Monthly charges** — high charges combined with low tenure is the riskiest combination
5. **Internet service** — fiber optic customers face stronger competitive alternatives

### Model Summary
- Random Forest outperforms Logistic Regression on AUC-ROC
- Both models meet the >0.75 AUC-ROC target
- The model can be used in production to score incoming customers and trigger retention workflows

### Next Steps
- Deploy the model via the Streamlit dashboard for real-time risk scoring
- A/B test retention campaigns on high-risk segments
- Retrain monthly as new customer data becomes available